# Afya MedQoL — exemplo de uso

Notebook interno com um exemplo de cálculo do índice Afya MedQoL usando a
biblioteca `afya_medqol` (calibração 2024_2).

Cobre:
1. Escoragem de um único respondente (dict de respostas).
2. Escoragem em lote a partir de um CSV.
3. Leitura das colunas de saída (theta, T-score).

## 1. Instalação e import

Se o pacote ainda não estiver instalado no ambiente:

In [1]:
# %pip install afya-medqol

In [2]:
import sys

sys.path.append("/Users/Marcela.Motta/Desktop/Projetos/afya_medqol/src")


In [3]:
import pandas as pd

from afya_medqol import MedQoLCalculator, ITENS_F1, ITENS_F2, ITENS_F3

print("Itens F1 (Qualidade de Vida):", ITENS_F1)
print("Itens F2 (Suporte Institucional):", ITENS_F2)
print("Itens F3 (Estresse Percebido):", ITENS_F3)

Itens F1 (Qualidade de Vida): ['F1_1_enjoymentoflife', 'F1_2_financialsufficiency', 'F1_3_accesstoinformation', 'F1_4_leisureopportunities', 'F1_5_mobilitypast2weeks', 'F1_6_accesstohealthservices']
Itens F2 (Suporte Institucional): ['F2_1_technicaltraining', 'F2_2_mentalhealthsupport', 'F2_3_coworkersupportnetwork', 'F2_4_educationalhandlingoferrors']
Itens F3 (Estresse Percebido): ['F3_1_stresshurtsperformance', 'F3_2_stressledtoerrors', 'F3_3_stresshurtsrelationships']


## 2. Escoragem de um único respondente

`MedQoLCalculator` precomputa a grade de quadratura e as probabilidades por
item uma única vez na construção — reutilize a mesma instância para escorar
vários respondentes/lotes.

In [4]:
calc = MedQoLCalculator()

respostas_medico = {
    "F1_1_enjoymentoflife": 4,
    "F1_2_financialsufficiency": 4,
    "F1_3_accesstoinformation": 3,
    "F1_4_leisureopportunities": 4,
    "F1_5_mobilitypast2weeks": 3,
    "F1_6_accesstohealthservices": 4,
    "F2_1_technicaltraining": 3,
    "F2_2_mentalhealthsupport": 3,
    "F2_3_coworkersupportnetwork": 2,
    "F2_4_educationalhandlingoferrors": 3,
    "F3_1_stresshurtsperformance": 4,
    "F3_2_stressledtoerrors": 3,
    "F3_3_stresshurtsrelationships": 4,
}

resultado = calc.score_physician(respostas_medico)

for chave in (
    "theta_F1", "theta_F2", "theta_F3", "theta_global", "T_score_global",
):
    print(f"{chave:28s} {resultado[chave]}")

theta_F1                     0.61802251746182
theta_F2                     0.16229324407529994
theta_F3                     0.11841287222597113
theta_global                 0.26718132559151314
T_score_global               52.67181325591513


## 3. Escoragem em lote (vários respondentes)

Um `DataFrame` com uma coluna por item (mesmos nomes de `ITENS_TODOS`) e uma
linha por respondente. Valores ausentes ou `999` (código de "não respondeu")
são tratados como omissos.

In [5]:
df_respostas = pd.DataFrame([
    {
        "medico_id": "M001",
        "F1_1_enjoymentoflife": 5, "F1_2_financialsufficiency": 5, "F1_3_accesstoinformation": 5,
        "F1_4_leisureopportunities": 5, "F1_5_mobilitypast2weeks": 5, "F1_6_accesstohealthservices": 5,
        "F2_1_technicaltraining": 5, "F2_2_mentalhealthsupport": 5,
        "F2_3_coworkersupportnetwork": 5, "F2_4_educationalhandlingoferrors": 5,
        "F3_1_stresshurtsperformance": 1, "F3_2_stressledtoerrors": 1, "F3_3_stresshurtsrelationships": 1,
    },
    {
        "medico_id": "M002",
        "F1_1_enjoymentoflife": 1, "F1_2_financialsufficiency": 1, "F1_3_accesstoinformation": 1,
        "F1_4_leisureopportunities": 1, "F1_5_mobilitypast2weeks": 1, "F1_6_accesstohealthservices": 1,
        "F2_1_technicaltraining": 1, "F2_2_mentalhealthsupport": 1,
        "F2_3_coworkersupportnetwork": 1, "F2_4_educationalhandlingoferrors": 1,
        "F3_1_stresshurtsperformance": 5, "F3_2_stressledtoerrors": 5, "F3_3_stresshurtsrelationships": 5,
    },
    {
        "medico_id": "M003",
        "F1_1_enjoymentoflife": 3, "F1_2_financialsufficiency": 3, "F1_3_accesstoinformation": 3,
        "F1_4_leisureopportunities": 3, "F1_5_mobilitypast2weeks": 3, "F1_6_accesstohealthservices": 3,
        "F2_1_technicaltraining": 3, "F2_2_mentalhealthsupport": 3,
        "F2_3_coworkersupportnetwork": 3, "F2_4_educationalhandlingoferrors": 3,
        "F3_1_stresshurtsperformance": 3, "F3_2_stressledtoerrors": 3, "F3_3_stresshurtsrelationships": 3,
    },
])

df_resultado = calc.calcular(df_respostas)
df_resultado[["medico_id", "theta_F1", "theta_F2", "theta_F3",
              "theta_global", "T_score_global"]]

,medico_id,theta_F1,theta_F2,theta_F3,theta_global,T_score_global
0,M001,2.541674,2.173227,-1.882549,2.240849,72.408492
1,M002,-3.436587,-1.520615,1.639552,-2.311715,26.882845
2,M003,-0.434366,0.275357,-0.418584,0.032791,50.327912


## 4. Salvando o resultado em CSV

`calcular_indice` é a função de conveniência: aceita um `DataFrame` ou um
caminho de CSV e opcionalmente já salva a saída.

In [6]:
from afya_medqol import calcular_indice

out = calcular_indice(df_respostas, caminho_saida="exemplo_scores.csv")
out.head()

,medico_id,F1_1_enjoymentoflife,F1_2_financialsufficiency,F1_3_accesstoinformation,F1_4_leisureopportunities,F1_5_mobilitypast2weeks,F1_6_accesstohealthservices,F2_1_technicaltraining,F2_2_mentalhealthsupport,F2_3_coworkersupportnetwork,...,F3_2_stressledtoerrors,F3_3_stresshurtsrelationships,theta_F1,theta_F2,theta_F3,theta_global,T_score_F1,T_score_F2,T_score_F3,T_score_global
0,M001,5,5,5,5,5,5,5,5,5,...,1,1,2.541674,2.173227,-1.882549,2.240849,75.416735,71.732275,31.174505,72.408492
1,M002,1,1,1,1,1,1,1,1,1,...,5,5,-3.436587,-1.520615,1.639552,-2.311715,15.634134,34.793850,66.395521,26.882845
2,M003,3,3,3,3,3,3,3,3,3,...,3,3,-0.434366,0.275357,-0.418584,0.032791,45.656344,52.753566,45.814163,50.327912


---

# IQoL — exemplo de uso (estudantes de medicina)

Índice IQoL (8 itens, modelo bifatorial) para estudantes de medicina.
Cobre os mesmos três casos: respondente único, lote e salvamento em CSV.

In [7]:
from afya_medqol import IQoLCalculator, ITENS_ESTUDANTE

print("Itens do IQoL:", ITENS_ESTUDANTE)

Itens do IQoL: ['F1_1_overallqol', 'F1_2_satisfactionwithhealth', 'F1_3_enjoymentoflife', 'F1_4_perceivedmeaninginlife', 'F2_1_energyfordailyactivities', 'F2_2_satisfactionwithsleep', 'F3_1_performdailyactivities', 'F3_2_capacityforwork']


## 1. Escoragem de um único estudante

In [8]:
calc_estudante = IQoLCalculator()

respostas_estudante = {
    "F1_1_overallqol": 4, "F1_2_satisfactionwithhealth": 4, "F1_3_enjoymentoflife": 4, "F1_4_perceivedmeaninginlife": 3,
    "F2_1_energyfordailyactivities": 3, "F2_2_satisfactionwithsleep": 4, "F3_1_performdailyactivities": 3, "F3_2_capacityforwork": 3,
}

resultado_estudante = calc_estudante.score_student(respostas_estudante)

for chave in (
    "theta_bem_estar_psicologico", "theta_vitalidade", "theta_capacidade_funcional", "theta_global",
    "T_score_global",
):
    print(f"{chave:42s} {resultado_estudante[chave]}")

theta_bem_estar_psicologico                0.26594480031808576
theta_vitalidade                           0.43833733308854916
theta_capacidade_funcional                 -0.5720210175706674
theta_global                               0.015143711762191245
T_score_global                             50.47216337332814


## 2. Escoragem em lote

In [9]:
df_estudantes = pd.DataFrame([
    {"aluno_id": "A001", "F1_1_overallqol": 5, "F1_2_satisfactionwithhealth": 5, "F1_3_enjoymentoflife": 5, "F1_4_perceivedmeaninginlife": 5,
     "F2_1_energyfordailyactivities": 5, "F2_2_satisfactionwithsleep": 5, "F3_1_performdailyactivities": 5, "F3_2_capacityforwork": 5},
    {"aluno_id": "A002", "F1_1_overallqol": 1, "F1_2_satisfactionwithhealth": 1, "F1_3_enjoymentoflife": 1, "F1_4_perceivedmeaninginlife": 1,
     "F2_1_energyfordailyactivities": 1, "F2_2_satisfactionwithsleep": 1, "F3_1_performdailyactivities": 1, "F3_2_capacityforwork": 1},
    {"aluno_id": "A003", "F1_1_overallqol": 3, "F1_2_satisfactionwithhealth": 3, "F1_3_enjoymentoflife": 3, "F1_4_perceivedmeaninginlife": 3,
     "F2_1_energyfordailyactivities": 3, "F2_2_satisfactionwithsleep": 3, "F3_1_performdailyactivities": 3, "F3_2_capacityforwork": 3},
])

df_resultado_estudantes = calc_estudante.calcular(df_estudantes)
df_resultado_estudantes[["aluno_id", "theta_bem_estar_psicologico", "theta_vitalidade",
                         "theta_capacidade_funcional", "theta_global", "T_score_global"]]

,aluno_id,theta_bem_estar_psicologico,theta_vitalidade,theta_capacidade_funcional,theta_global,T_score_global
0,A001,0.546408,0.036023,0.311684,0.380536,61.294682
1,A002,-1.056184,0.050283,-0.254782,-0.598458,32.297937
2,A003,-0.617983,0.019531,-0.081657,-0.329279,40.270727


## 3. Salvando o resultado em CSV

In [10]:
from afya_medqol import calcular_indice_estudante

out_estudante = calcular_indice_estudante(df_estudantes, caminho_saida="exemplo_scores_iqol.csv")
out_estudante.head()

,aluno_id,F1_1_overallqol,F1_2_satisfactionwithhealth,F1_3_enjoymentoflife,F1_4_perceivedmeaninginlife,F2_1_energyfordailyactivities,F2_2_satisfactionwithsleep,F3_1_performdailyactivities,F3_2_capacityforwork,theta_bem_estar_psicologico,theta_vitalidade,theta_capacidade_funcional,theta_global,T_score_global
0,A001,5,5,5,5,5,5,5,5,0.546408,0.036023,0.311684,0.380536,61.294682
1,A002,1,1,1,1,1,1,1,1,-1.056184,0.050283,-0.254782,-0.598458,32.297937
2,A003,3,3,3,3,3,3,3,3,-0.617983,0.019531,-0.081657,-0.329279,40.270727
